# Hands on Preaparin Analysis Datasets

In this code I perform the World Bank's practical task for data contruction, the third phase of a research proyect.

## Set environtment parameters

In [1]:
# Libraries
import pandas as pd
import numpy as np

In [2]:
# Paths
interim_path = '../data/interim'
raw_path = '../data/raw'
processed_path = '../data/processed'

In [34]:
# Params
conversion_rate = 0.00039 # USD-(mog)currency

## Load information

In [ ]:
# Load treatment status
treat_status = pd.read_csv(f'{raw_path}/treat_status.csv')

# Load TZA cash-conditional transfer.
tza_cct = pd.read_csv(f'{interim_path}/TZA_CCT_HH.csv')
print(tza_cct.columns)
print('\n')

# Looking for unique values on farm area
print("Unique area values:\n", tza_cct['ar_farm_unit'].unique())
print('\n')

# Looking for currency values
print("Currency description:\n", tza_cct[['food_cons', 'nonfood_cons']].describe())
print('\n')

Index(['vid', 'treatment', 'district', 'district_name'], dtype='str')


Index(['vid', 'hhid', 'enid', 'floor', 'roof', 'walls', 'water', 'enegry',
       'rel_head', 'female_head', 'hh_size', 'n_child_5', 'n_child_17',
       'n_adult', 'n_elder', 'read', 'sick', 'food_cons', 'nonfood_cons',
       'farm', 'ar_farm', 'ar_farm_unit', 'crop', 'crop_other', 'crop_prp',
       'livestock_now', 'livestock_before', 'drought_flood', 'crop_damage',
       'trust_mem', 'trust_lead', 'assoc', 'health', 'duration',
       'submissionday', 'key'],
      dtype='str')


Unique area values:
 <ArrowStringArray>
['Acre', 'Hectare', nan]
Length: 3, dtype: str


Currency description:
           food_cons  nonfood_cons
count  1.758000e+03  1.758000e+03
mean   6.392233e+05  1.468731e+05
std    5.131717e+05  2.510197e+05
min    5.096000e+03  0.000000e+00
25%    2.616250e+05  2.347750e+04
50%    5.174000e+05  6.993300e+04
75%    8.580000e+05  1.665750e+05
max    4.903600e+06  4.108400e+06




### Exercise 1: Standardize unos for land and currency

In [35]:
# Standardize / Homogenize variables
# Area values
tza_cct['area_acre'] = np.where(tza_cct['ar_farm_unit'] == 'Hectare',
                                tza_cct['ar_farm'] * 2.74, tza_cct['ar_farm'])


# Currency values
tza_cct[['food_cons_usd', 'nonfood_cons_usd']] = tza_cct[['food_cons', 'nonfood_cons']] * conversion_rate

# Looking for unique values on farm area
print("Currency description:\n", tza_cct['area_acre'].describe())
print('\n')

# Looking for currency values
print("Currency description:\n", tza_cct[['food_cons_usd', 'nonfood_cons_usd']].describe())
print('\n')


Currency description:
 count    1620.000000
mean        2.118211
std         2.973191
min         0.125000
25%         1.000000
50%         1.500000
75%         2.550384
max        88.709025
Name: area_acre, dtype: float64


Currency description:
        food_cons_usd  nonfood_cons_usd
count    1758.000000       1758.000000
mean      249.297099         57.280503
std       200.136948         97.897676
min         1.987440          0.000000
25%       102.033750          9.156225
50%       201.786000         27.273870
75%       334.620000         64.964250
max      1912.404000       1602.276000




### Excersice 2: Deal with Outliers

In [36]:
# Windsorize outliers

windsor_vars = ['area_acre','food_cons_usd', 'nonfood_cons_usd']

for var in windsor_vars:
    # Extract 5th and 95th percentiles simultaneously
    p_05, p_95 = tza_cct[var].quantile([0.05, 0.95])
    
    print(f"5th percentile-{var}: {p_05}.\n95th percentile-{var}: {p_95}.\n\n")
    
    # Apply windsorization
    tza_cct[var] = np.where( tza_cct[var] < p_05, 
                            p_05, tza_cct[var])
    tza_cct[var] = np.where( tza_cct[var] > p_95, 
                            p_95, tza_cct[var])
    del p_05, p_95



5th percentile-area_acre: 0.25.
95th percentile-area_acre: 5.544314042897625.


5th percentile-food_cons_usd: 40.43832000000001.
95th percentile-food_cons_usd: 638.8656299999999.


5th percentile-nonfood_cons_usd: 0.585.
95th percentile-nonfood_cons_usd: 206.96129999999957.




### Excercise 3: Merge data

In [ ]:
# Exploring tratment status dataset
treat_status.head()

,vid,treatment,district,district_name
0,1,0,2,Bagamoyo
1,2,1,2,Bagamoyo
2,3,1,2,Bagamoyo
3,4,0,2,Bagamoyo
4,5,0,2,Bagamoyo


In [40]:
# Looking for variables key for merging
print(treat_status.columns, treat_status.shape)
print('\n')

print(tza_cct.columns, tza_cct.shape)
print('\n')

# Merge dataframes
final_data = treat_status.merge(tza_cct, how = 'left', on = 'vid')

# Reaordering columns
cols_to_front = ['vid','hhid', 'enid']

# Sort the remaining columns
remaining_cols = [col for col in final_data.columns if col not in cols_to_front] # use sorted() if want it ordered alphabetically

# Apply the new order
final_data = final_data[cols_to_front + remaining_cols]

# Sort data by id values
final_data = final_data.sort_values(by = ['vid','hhid', 'enid'])

# Save results
final_data.to_csv(f'{processed_path}/data_for_analysis.csv', index = False)

Index(['vid', 'treatment', 'district', 'district_name'], dtype='str') (80, 4)


Index(['vid', 'hhid', 'enid', 'floor', 'roof', 'walls', 'water', 'enegry',
       'rel_head', 'female_head', 'hh_size', 'n_child_5', 'n_child_17',
       'n_adult', 'n_elder', 'read', 'sick', 'food_cons', 'nonfood_cons',
       'farm', 'ar_farm', 'ar_farm_unit', 'crop', 'crop_other', 'crop_prp',
       'livestock_now', 'livestock_before', 'drought_flood', 'crop_damage',
       'trust_mem', 'trust_lead', 'assoc', 'health', 'duration',
       'submissionday', 'key', 'area_acre', 'food_cons_usd',
       'nonfood_cons_usd'],
      dtype='str') (1758, 39)


